In [0]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("SQLServerLab").getOrCreate()


In [0]:
# Connection properties for SQL Server
jdbcHostname = "ludsampledb.database.windows.net"
jdbcDatabase = "SampleDB"
jdbcPort = 1433
jdbcUsername = "sampleuser"
jdbcPassword = "3]bP82+X"

# Define the JDBC URL
jdbcUrl = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};database={jdbcDatabase}"

# Connection properties
connectionProperties = {
    "user" : jdbcUsername,
    "password" : jdbcPassword,
    "driver" : "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [0]:
# Query data from the SQL Server table "Customer"
customers_df = spark.read.jdbc(url=jdbcUrl, table="Customer", properties=connectionProperties)

# Show the data retrieved from the Customer table
customers_df.show()

+--------------------+-----+----------+-----+-----------+
|             Address|  Zip|      Name|State|       City|
+--------------------+-----+----------+-----+-----------+
|      84 Arcadia Dr.|44145|John Smith|   OH|   Westlake|
|8691 Rockledge St...|12302|  Jane Doe|   NY|Schenectady|
|     6 Tallwood Road|97402|Tom Tucker|   OR|     Eugene|
|     37 Chestnut Rd.|48150| Jack Hill|   MI|    Livonia|
|909 Hidden Pond Road|44145|Mark Stone|   OH|   Westlake|
+--------------------+-----+----------+-----+-----------+



In [0]:
# Create a temporary view for querying
customers_df.createOrReplaceTempView("customers")

# Query 1: Retrieve all customers from Ohio (OH)
ohio_customers = spark.sql("""
    SELECT Name, Address, City, Zip 
    FROM customers 
    WHERE State = 'OH'
""")

# Show the result
print("Customers from Ohio:")
ohio_customers.show()

Customers from Ohio:
+----------+--------------------+--------+-----+
|      Name|             Address|    City|  Zip|
+----------+--------------------+--------+-----+
|John Smith|      84 Arcadia Dr.|Westlake|44145|
|Mark Stone|909 Hidden Pond Road|Westlake|44145|
+----------+--------------------+--------+-----+



In [0]:
# Query 2: Count customers by city
customers_by_city = spark.sql("""
    SELECT City, COUNT(*) as CustomerCount
    FROM customers
    GROUP BY City
    ORDER BY CustomerCount DESC
""")

# Show the result
print("Number of Customers by City:")
customers_by_city.show()

Number of Customers by City:
+-----------+-------------+
|       City|CustomerCount|
+-----------+-------------+
|   Westlake|            2|
|Schenectady|            1|
|    Livonia|            1|
|     Eugene|            1|
+-----------+-------------+



In [0]:
# Query 3: Filter customers by Zip code range (44000 to 45000)
zip_range_customers = spark.sql("""
    SELECT Name, Zip, State
    FROM customers
    WHERE Zip BETWEEN 44000 AND 45000
    ORDER BY Zip
""")

# Show the result
print("Customers with Zip codes between 44000 and 45000:")
zip_range_customers.show()

Customers with Zip codes between 44000 and 45000:
+----------+-----+-----+
|      Name|  Zip|State|
+----------+-----+-----+
|John Smith|44145|   OH|
|Mark Stone|44145|   OH|
+----------+-----+-----+

